# Quickstart: Training pinnrl on a `The Well` benchmark dataset

This notebook walks through training a neural surrogate on `active_matter` from [Polymathic AI's `the_well`](https://github.com/PolymathicAI/the_well). The Well dataset has no analytical solver in pinnrl, so we use the **`data_only`** mode — the network learns to reproduce the reference field directly, without a PDE residual term.

Prerequisites:

```bash
pip install 'pinnrl[well]'
```

If you have already downloaded the dataset locally with `the-well-download`, point `base=` at it. Otherwise the loader streams from `hf://datasets/polymathic-ai/`.

## 1. Inspect the registry

In [1]:
from pinnrl.datasets import WELL_REGISTRY, get_entry

for name, entry in sorted(WELL_REGISTRY.items()):
    print(f"{name:35s}  {entry.n_spatial_dims}D  mode={entry.recommended_mode:14s}  arch={entry.default_architecture}")

entry = get_entry('active_matter')
print('\nactive_matter fields:', entry.fields)
print('domain:', entry.domain, 'time:', entry.time_domain)

MHD_64                               3D  mode=data_only       arch=mlp
acoustic_scattering_maze             2D  mode=data_augmented  arch=fno
active_matter                        2D  mode=data_only       arch=fno
euler_multi_quadrants_periodicBC     2D  mode=data_only       arch=fno
gray_scott_reaction_diffusion        2D  mode=data_only       arch=fno
helmholtz_staircase                  2D  mode=data_augmented  arch=fno
planetswe                            2D  mode=data_only       arch=fno
rayleigh_benard                      2D  mode=data_only       arch=fno
rayleigh_taylor_instability          3D  mode=data_only       arch=mlp
shear_flow                           2D  mode=data_only       arch=fno
turbulent_radiative_layer_2D         2D  mode=data_only       arch=fno
viscoelastic_instability             2D  mode=data_only       arch=fno

active_matter fields: ('concentration', 'velocity_x', 'velocity_y', 'orientation_xx', 'orientation_xy')
domain: ((0.0, 1.0), (0.0, 1.0)) time: (0.0

## 2. Load a small slice into memory

`load_well_slice` flattens the trajectory grid into the same `(x, t, u)` triple pinnrl uses everywhere — so the rest of the framework treats it identically to synthetic observations.

In [2]:
from pinnrl.datasets import load_well_slice

slice_ = load_well_slice(
    name='active_matter',
    split='train',
    n_traj=2,
    n_points=5_000,
    seed=0,
    device='cpu',
)
for k, v in slice_.items():
    print(f'{k}: shape={tuple(v.shape)}  dtype={v.dtype}')

TheWellNotInstalledError: The Well datasets require the optional dependency. Install with:
    pip install 'pinnrl[well]'
or:
    pip install the_well h5py huggingface-hub

## 3. Train an FNO in data_only mode

The fastest path is the CLI: `build_config_dict` overlays the registry defaults and the trainer takes it from there.

In [ ]:
import yaml, torch
from pathlib import Path
from pinnrl.training.train import build_config_dict, run_training

yaml_cfg = yaml.safe_load(Path('../pinnrl/config/config.yaml').read_text())
yaml_cfg.setdefault('training', {})['num_epochs'] = 200

config_dict = build_config_dict(
    yaml_cfg,
    pde_name='Heat Equation',  # Placeholder — overridden by dataset defaults.
    arch_type='fno',
    use_rl=False,
    epochs=200,
    dataset={
        'name': 'active_matter',
        'split': 'train',
        'n_traj': 2,
        'n_points': 5_000,
        'seed': 0,
        'base': None,
        'use_defaults': True,
    },
)
print('mode:', config_dict['training']['mode'])
print('input_dim/output_dim:', config_dict['model']['input_dim'], '/', config_dict['model']['output_dim'])

device = torch.device('cpu')
config_dict['device'] = str(device)
run_training(config_dict, device)

## 4. Plot prediction vs. reference

After training completes, the latest experiment directory under `results_dir` contains `final_model.pt` and a `live_snapshot.npz`. Reload them and compare a single field channel.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

results_dir = Path(yaml_cfg['paths']['results_dir'])
experiment = sorted(results_dir.glob('*active_matter*'))[-1]
snapshot = np.load(experiment / 'live_snapshot.npz')
u_pred = snapshot['u_pred']
u_ref = snapshot.get('u_ref', np.zeros_like(u_pred))

fig, axes = plt.subplots(1, 2, figsize=(10, 4))
axes[0].imshow(u_pred, cmap='viridis')
axes[0].set_title('Prediction (channel 0)')
axes[1].imshow(u_ref, cmap='viridis')
axes[1].set_title('Reference')
plt.tight_layout()
plt.show()

## 5. Try a matched dataset (`acoustic_scattering_maze`)

For datasets whose default PDE matches an existing pinnrl solver, switch to `data_augmented` to combine the residual loss with the dataset's reference field. This is the regime where pinnrl's RL adaptive sampler can be benchmarked against the ground truth from The Well.

In [ ]:
matched = build_config_dict(
    yaml_cfg,
    pde_name='Wave Equation',
    arch_type='fno',
    use_rl=True,
    epochs=300,
    dataset={
        'name': 'acoustic_scattering_maze',
        'split': 'train',
        'n_traj': 1,
        'n_points': 4_000,
        'seed': 0,
        'base': None,
        'use_defaults': True,
    },
)
print('mode:', matched['training']['mode'])  # data_augmented
print('observation_data source:', matched['pde']['observation_data']['source'])